# TOML - JavaScript

All 5 JavaScript examples from [docs/toml.md](https://platob.github.io/yggdryl/toml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

## Raw shared-Scalar access

In [ ]:
const assert = require('node:assert/strict')
const { Scalar, toml } = require('yggdryl')

const source = 'title = "yggdryl"\ncount = 3\n\n[owner]\nname = "Ada"\n'
const natural = toml.loads(source)
const value = toml.loads(source, { scalar: true })
const encoded = toml.dumps(value)

assert.ok(value instanceof Scalar)
assert.equal(value.kind, 'record')
assert.deepEqual(value.asJs(), natural)
assert.ok(Buffer.isBuffer(encoded))
assert.deepEqual(toml.loads(encoded), natural)

## Natural values and exact Fields

In [ ]:
const assert = require('node:assert/strict')
const { fields, toml } = require('yggdryl')

const row = fields.struct(
  'row',
  [fields.decimal128('amount', 8, 2, { nullable: false })],
  { nullable: false },
)
const decoded = toml.loads("amount = '12.50'\n", { field: row })

assert.equal(decoded.amount.kind, 'd128')
assert.equal(decoded.amount.unscaled, 1250n)

## Documents and streams

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { pathToFileURL } = require('node:url')
const { toml } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-toml-'))
const target = path.join(root, 'value.toml')
toml.dump({ id: 1 }, target)

assert.deepEqual(toml.load(pathToFileURL(target)), { id: 1 })
fs.rmSync(root, { recursive: true, force: true })

## Formatting

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

const value = { items: [1, 2, 3] }
const laidOut = toml.dumps(value, { indent: 2 })
const compact = toml.dumps(value, { indent: null })

assert.notDeepEqual(laidOut, compact)
assert.deepEqual(toml.loads(laidOut), toml.loads(compact))

## Placeholders

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

const document = 'host = "{{ HOST }}"\nport = "{{ PORT }}"\n'
const value = toml.loads(document, {
  placeholders: { HOST: 'db.internal', PORT: 5432 },
})

assert.deepEqual(value, { host: 'db.internal', port: 5432 })